In [3]:
# Air Conditioner Control - Mamdani AND Sugeno FIS
# Rules based on the image (16 rules)

# Triangular membership function
def trimf(x, a, b, c):
    if x <= a or x >= c:
        return 0.0
    elif a < x <= b:
        return (x - a) / (b - a)
    else:
        return (c - x) / (c - b)

In [4]:
# ── TEMPERATURE memberships (°C) ──────────────────────
def temp_mf(T):
    return {
        'very_low':  trimf(T,  0, 10, 20),
        'low':       trimf(T, 10, 20, 30),
        'high':      trimf(T, 20, 30, 40),
        'very_high': trimf(T, 30, 40, 50)
    }

# ── HUMIDITY memberships (%) ──────────────────────────
def hum_mf(H):
    return {
        'dry':         trimf(H,  0, 10, 30),
        'comfortable': trimf(H, 20, 40, 60),
        'humid':       trimf(H, 50, 65, 80),
        'sticky':      trimf(H, 70, 85, 100)
    }

In [5]:
# ── 16 RULES from image ───────────────────────────────
def apply_rules(t, h):
    rules = [
        (t['very_low'],  h['dry'],         'off'),
        (t['very_low'],  h['comfortable'], 'off'),
        (t['very_low'],  h['humid'],       'off'),
        (t['very_low'],  h['sticky'],      'low'),
        (t['low'],       h['humid'],       'low'),
        (t['low'],       h['comfortable'], 'off'),
        (t['low'],       h['humid'],       'low'),
        (t['low'],       h['sticky'],      'medium'),
        (t['high'],      h['dry'],         'low'),
        (t['high'],      h['comfortable'], 'medium'),
        (t['high'],      h['humid'],       'fast'),
        (t['high'],      h['sticky'],      'fast'),
        (t['very_high'], h['dry'],         'medium'),
        (t['very_high'], h['comfortable'], 'fast'),
        (t['very_high'], h['humid'],       'fast'),
        (t['very_high'], h['sticky'],      'fast'),
    ]
    # Group by output: take max firing strength per output category
    output = {'off': 0, 'low': 0, 'medium': 0, 'fast': 0}
    for t_val, h_val, label in rules:
        strength = min(t_val, h_val)   # AND = min
        output[label] = max(output[label], strength)
    return output

In [9]:
# ══ MAMDANI ══════════════════════════════════════════
def mamdani(T, H):
    t = temp_mf(T)
    h = hum_mf(H)
    fired = apply_rules(t, h)

    # Output MFs for speed (0-100 RPM scale)
    speed_mf = {
        'off':    (0,   0,  20),
        'low':    (10,  30, 50),
        'medium': (40,  60, 80),
        'fast':   (70,  90, 100)
    }

    # Defuzzify using weighted average (centroid approx)
    numerator = 0
    denominator = 0
    for label, strength in fired.items():
        a, b, c = speed_mf[label]
        centroid = b  # peak of triangle as representative value
        numerator   += strength * centroid
        denominator += strength

    result = numerator / denominator if denominator != 0 else 0
    print(f"\n── Mamdani ──")
    print(f"Rule firing strengths : {fired}")
    print(f"Defuzzified Speed     : {result:.2f} (0-100 scale)")
    return result

In [10]:
# ══ SUGENO ═══════════════════════════════════════════
def sugeno(T, H):
    t = temp_mf(T)
    h = hum_mf(H)
    fired = apply_rules(t, h)

    # Sugeno uses crisp constant output values (no output MF needed)
    crisp = {'off': 0, 'low': 25, 'medium': 50, 'fast': 100}

    numerator   = sum(fired[l] * crisp[l] for l in fired)
    denominator = sum(fired.values())

    result = numerator / denominator if denominator != 0 else 0
    print(f"\n── Sugeno ──")
    print(f"Rule firing strengths : {fired}")
    print(f"Defuzzified Speed     : {result:.2f} (0-100 scale)")
    return result

In [11]:
# ── USER INPUT ────────────────────────────────────────
T = float(input("Enter Temperature (°C, e.g. 35): "))
H = float(input("Enter Humidity (%, e.g. 70): "))

m = mamdani(T, H)
s = sugeno(T, H)

print(f"\n── Comparison ──")
print(f"Mamdani Speed : {m:.2f}")
print(f"Sugeno Speed  : {s:.2f}")
print(f"Difference    : {abs(m-s):.2f}")


── Mamdani ──
Rule firing strengths : {'off': 0, 'low': 0, 'medium': 0, 'fast': 0.6666666666666666}
Defuzzified Speed     : 90.00 (0-100 scale)

── Sugeno ──
Rule firing strengths : {'off': 0, 'low': 0, 'medium': 0, 'fast': 0.6666666666666666}
Defuzzified Speed     : 100.00 (0-100 scale)

── Comparison ──
Mamdani Speed : 90.00
Sugeno Speed  : 100.00
Difference    : 10.00
